# 🦙 Phase 1: QLoRA Fine-Tuning — LLaMA 3.2 3B

Multi-task instruction-following model (Summarization + Chat + Persona)

**⚠️ Before running: Runtime → Change runtime type → T4 GPU**

## 1 · Install Dependencies

In [ ]:
!pip install -q -U pip
!pip install -q -U \
    transformers \
    peft \
    trl \
    datasets \
    accelerate \
    bitsandbytes

print("\n✅ Packages installed! Now RESTART the runtime:")
print("   Runtime → Restart runtime  (or Ctrl+M .)")

## 2 · Verify GPU & Login

In [ ]:
import torch
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

import transformers, peft, trl, bitsandbytes
print(f"transformers: {transformers.__version__}")
print(f"peft:         {peft.__version__}")
print(f"trl:          {trl.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print("\n✅ GPU ready!")

In [ ]:
from huggingface_hub import login
login()  # Paste your HF token (needs access to meta-llama/Llama-3.2-3B)

## 3 · Upload Datasets

Upload your Phase 0 processed files to Colab (they'll appear in `/content/`):
- `summarization_prepared.jsonl`
- `persona_prepared.jsonl`
- `chat_prepared.jsonl` *(optional — gracefully skipped if not present)*

In [ ]:
import os
import shutil

UPLOAD_DIR = "/content"
DATA_DIR = "/content/data/processed"
os.makedirs(DATA_DIR, exist_ok=True)

expected_files = [
    "summarization_prepared.jsonl",
    "persona_prepared.jsonl",
    "chat_prepared.jsonl",
]

found = []
for f in expected_files:
    src = os.path.join(UPLOAD_DIR, f)
    dst = os.path.join(DATA_DIR, f)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        lines = sum(1 for _ in open(dst, 'r', encoding='utf-8'))
        size_mb = os.path.getsize(dst) / 1024**2
        print(f"  ✅ {f}: {lines:,} samples ({size_mb:.1f} MB)")
        found.append(f)
    elif os.path.exists(dst):
        lines = sum(1 for _ in open(dst, 'r', encoding='utf-8'))
        size_mb = os.path.getsize(dst) / 1024**2
        print(f"  ✅ {f}: {lines:,} samples ({size_mb:.1f} MB) [already in place]")
        found.append(f)
    else:
        print(f"  ⏭️  {f}: not found (will be skipped)")

if not found:
    print("\n⚠️  No datasets found! Upload files first.")
else:
    print(f"\n✅ {len(found)} dataset(s) ready!")

## 4 · Load Base Model (8-bit Quantized)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "meta-llama/Llama-3.2-3B"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype="auto",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"\n✅ Model loaded: {MODEL_ID}")
print(f"   Vocab size: {len(tokenizer):,}")
print(f"   Model dtype: 8-bit quantized")

## 5 · Attach LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=6,
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print("\n✅ LoRA adapters attached (base model frozen)")

## 6 · Load & Prepare Datasets

In [ ]:
import json
from datasets import Dataset, DatasetDict

DATA_DIR = "/content/data/processed"

dataset_files = [
    "summarization_prepared.jsonl",
    "persona_prepared.jsonl",
    "chat_prepared.jsonl",
]

all_records = []
for fname in dataset_files:
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        print(f"  ⏭️  Skipped (not found): {fname}")
        continue
    with open(fpath, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]
    all_records.extend(records)
    print(f"  ✅ Loaded {len(records):,} samples from {fname}")

print(f"\n📊 Total merged samples: {len(all_records):,}")

merged_dataset = Dataset.from_list(all_records)
print(f"   Columns: {merged_dataset.column_names}")

In [ ]:
def format_instruction(example):
    instruction = example["instruction"].strip()
    input_text = example["input"].strip()
    output_text = example["output"].strip()

    if input_text:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n{output_text}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{output_text}"
        )
    return {"text": text}

formatted_dataset = merged_dataset.map(
    format_instruction,
    remove_columns=merged_dataset.column_names,
    desc="Formatting",
)

split = formatted_dataset.train_test_split(test_size=0.1, seed=42)
dataset_dict = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
})

print(f"✅ Train: {len(dataset_dict['train']):,} | Val: {len(dataset_dict['validation']):,}")
print(f"\n📋 Sample:")
print(dataset_dict["train"][0]["text"][:400])

## 7 · Sample Generation Callback

In [ ]:
from transformers import TrainerCallback

SAMPLE_PROMPTS = [
    {
        "task": "Summarization",
        "text": (
            "### Instruction:\n"
            "Summarize the following scientific document:\n\n"
            "### Input:\n"
            "Deep learning has achieved remarkable success in NLP, "
            "computer vision, and speech recognition. Recent advances "
            "in transformer architectures have enabled models to capture "
            "long-range dependencies more effectively.\n\n"
            "### Response:\n"
        ),
    },
    {
        "task": "Persona (Chandler)",
        "text": (
            "### Instruction:\n"
            "Reply as Chandler Bing from Friends:\n\n"
            "### Input:\n"
            "Do you want to go to the gym with me?\n\n"
            "### Response:\n"
        ),
    },
]


class SampleGenerationCallback(TrainerCallback):
    def __init__(self, every_n_steps=200):
        self.every_n_steps = every_n_steps

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every_n_steps != 0 or state.global_step == 0:
            return
        mdl = kwargs.get("model", None)
        if mdl is None:
            return
        mdl.eval()
        print(f"\n{'='*60}")
        print(f"  📝 Sample Generations @ Step {state.global_step}")
        print(f"{'='*60}")
        for sample in SAMPLE_PROMPTS:
            inputs = tokenizer(sample["text"], return_tensors="pt", truncation=True, max_length=512).to(mdl.device)
            with torch.no_grad():
                outputs = mdl.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.1)
            generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
            response = generated[len(sample["text"]):].strip()
            print(f"\n  [{sample['task']}]")
            print(f"  → {response[:300]}")
        print(f"{'='*60}\n")
        mdl.train()


sample_callback = SampleGenerationCallback(every_n_steps=200)
print("✅ Callback ready (generates samples every 200 steps)")

## 8 · Train 🚀

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./out",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,
    logging_steps=100,
    save_steps=500,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=200,
    max_grad_norm=0.3,
    report_to="none",
    seed=42,
    max_seq_length=512,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["validation"],
    processing_class=tokenizer,
    callbacks=[sample_callback],
)

model.config.use_cache = False

print(f"🚀 Ready to train!")
print(f"   Train: {len(dataset_dict['train']):,} | Val: {len(dataset_dict['validation']):,}")
print(f"   Epochs: {training_args.num_train_epochs} | LR: {training_args.learning_rate}")

In [ ]:
trainer.train()
print("\n✅ Training complete!")

## 9 · Save LoRA Adapter + Tokenizer

In [ ]:
OUTPUT_DIR = "/content/out"

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Verify key files
import os
for name in ["adapter_model.safetensors", "adapter_model.bin", "adapter_config.json"]:
    path = os.path.join(OUTPUT_DIR, name)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f"  ✅ {name} ({size_mb:.2f} MB)")

print(f"\n✅ Saved to {OUTPUT_DIR}/")

In [ ]:
!ls -lh /content/out/

## 10 · Training Summary & Quick Test

In [ ]:
metrics = trainer.state.log_history
train_losses = [(m["step"], m["loss"]) for m in metrics if "loss" in m and "eval_loss" not in m]
eval_losses = [(m["step"], m["eval_loss"]) for m in metrics if "eval_loss" in m]

print("📊 Training Log:")
print(f"   Final train loss: {train_losses[-1][1]:.4f}" if train_losses else "   No train losses")
print(f"   Final eval loss:  {eval_losses[-1][1]:.4f}" if eval_losses else "   No eval losses")
print(f"   Total steps: {trainer.state.global_step}")
print(f"   Epochs: {trainer.state.epoch:.1f}")

In [ ]:
model.eval()
test_prompts = [
    "### Instruction:\nSummarize the following conversation:\n\n### Input:\nAlice: Hey, are we meeting at 3?\nBob: Yes! Same place.\nAlice: Perfect.\n\n### Response:\n",
    "### Instruction:\nReply as Chandler Bing from Friends:\n\n### Input:\nI just got a promotion!\n\n### Response:\n",
]

print("🧪 Quick Inference Test:\n")
for i, prompt in enumerate(test_prompts, 1):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()
    print(f"Test {i}: {response[:200]}\n")

## 11 · Download Checkpoint

In [ ]:
!cd /content && zip -r /content/llama3_lora_phase1.zip out/
!ls -lh /content/llama3_lora_phase1.zip
print("\n✅ Ready for download!")

In [ ]:
try:
    from google.colab import files
    files.download('/content/llama3_lora_phase1.zip')
    print('📥 Download started!')
except ImportError:
    print('Download /content/llama3_lora_phase1.zip from Files panel.')